# Student Admission Chance Predictor Training Notebook

This notebook loads `../data/student-data.csv`, prepares the data, trains a `RandomForestClassifier` pipeline with one-hot encoding, evaluates it, and saves the model to `../model/model.pkl`.

In [ ]:
from pathlib import Path
import sys

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().resolve().parent
BACKEND_DIR = PROJECT_ROOT / 'backend'
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from preprocess import (
    ADMITTED_LABEL,
    CATEGORICAL_FEATURES,
    FEATURE_COLUMNS,
    NEEDS_IMPROVEMENT_LABEL,
    TARGET_COLUMN,
    build_preprocessing_pipeline,
    build_training_data,
)

df = pd.read_csv('../data/student-data.csv')
X, y = build_training_data(df)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

pipeline = Pipeline(
    steps=[
        ('preprocess', build_preprocessing_pipeline()),
        ('model', RandomForestClassifier(n_estimators=200, random_state=42)),
    ]
)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
matrix = confusion_matrix(y_test, y_pred, labels=[ADMITTED_LABEL, NEEDS_IMPROVEMENT_LABEL])

print('Features used:', FEATURE_COLUMNS)
print('Categorical features:', CATEGORICAL_FEATURES)
print('Target column:', TARGET_COLUMN)
print(f'Accuracy: {accuracy:.2f}')
print('Confusion matrix:')
print(matrix)
print('\nClassification report:')
print(classification_report(y_test, y_pred))

model_dir = PROJECT_ROOT / 'model'
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / 'model.pkl'
joblib.dump(pipeline, model_path)
model_path